# Brain-Age Pipeline — Methodology

This notebook contains the full **methodology** stage of the pipeline (SFCN training through the
longitudinal / clinical-conversion analysis), following *"Isolating Biological Aging from Scanner
Noise: A Reliability Framework for Brain-Age Trajectories"*. It picks up where `preprocessing.ipynb`
leaves off: it expects `T1 preprocessed/` to already contain the 6-step-preprocessed `_preproc.npz`
tensors.

**What's preserved from the original run:** Cells 6–17 below are unchanged from the session that was
already executed on the reported HPC (Intel Core Ultra 9 285K, 128GB RAM, RTX 5080 16GB) — their
printed outputs (validation MAE, test MAE/RMSE, the 71.3% derived age-group classification accuracy,
the measurement-noise floor, etc.) are kept in place exactly as produced, so this notebook is both a
runnable pipeline and a record of the results already obtained.

**What's fixed:** Cells 18 and 19 (longitudinal trajectories, signal isolation, and survival /
continuous-endpoint analysis) had not been successfully executed — see the "Errors found & fixed"
note directly above CELL 18 for the three bugs found and how they were corrected. They are included
here as runnable, self-contained cells but their outputs are empty because they have not been run
yet in this environment.

## Paper section map

| Cells | Paper section |
|---|---|
| 6–9 | III-A Data partitioning, III-C.1–2 model architecture setup |
| 10–11 | III-C.2 training (SFCN ensemble, GPU/AMP, HPC config) |
| 12–13 | III-C.4 ensemble validation & held-out test inference |
| 14 | III-B systemic age-bias correction, BAG definition |
| 15 | Derived age-group analysis (not in the model's training objective; a diagnostic view of BAG) |
| 16 | III-E.2 same-day repeat noise floor (σ_SEM, MDC) |
| 17 | Consolidated final statistics |
| 18 | III-D longitudinal trajectories (LME), III-E.3–4 noise stratification & signal isolation, III-F/G Cox PH survival |
| 19 | III-G.6 pre-registered continuous clinical endpoint (CDR-SUM / MMSE slopes) |


In [ ]:
# =========================================================
# ENVIRONMENT PROBE
# =========================================================
# Quick sanity-check that confirms:
#   • which machine (hostname) we are running on — important because the
#     notebook was originally executed on an HPC node (i9 / RTX 5080) and
#     may also be opened on a local Windows workstation.
#   • the current working directory so that all relative paths further down
#     (./UKBiobank_deep_pretrain-master/, ./checkpoints_v2/, etc.) resolve
#     correctly.
# Run this cell first after every kernel restart before executing any other
# cell.

import socket
import os

print("Hostname:", socket.gethostname())
print("Working directory:", os.getcwd())

In [1]:
# =========================================================
# GPU / PYTORCH VERIFICATION
# =========================================================
# Sets the global `device` that every subsequent cell uses for tensor
# allocation and model placement.
#
# Checks performed:
#   1. torch.cuda.is_available()  — confirms a CUDA-capable GPU is present.
#   2. torch.cuda.get_device_name(0)  — prints the exact GPU model so we can
#      verify the expected RTX 5080 16 GB (Blackwell) is being used.
#   3. Existence of the UKBiobank pretrained SFCN weights file — the whole
#      pipeline depends on this checkpoint, so it is better to catch a missing
#      file here than deep inside Cell 11.
#
# If `device` is "cpu", training will proceed but will be orders of magnitude
# slower; the dataset-caching strategy in Cell 7 still works because it uses
# system RAM, not GPU memory.

import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))
import torch; print(torch.__version__); print(torch.cuda.is_available())

import os
print(os.getcwd())
print(os.path.exists("./UKBiobank_deep_pretrain-master/brain_age/run_20190719_00_epoch_best_mae.p"))

Device: cuda
GPU: NVIDIA GeForce RTX 5080
2.14.0+cu130
True
/mnt/d/brain age prep and meth/v2 approach
True


## CELL 6 — SFCN architecture
Defines the same 3D SFCN architecture used by the training script and instantiates one model to verify the forward pass.


In [2]:
# =========================================================
# CELL 6 — SFCN ARCHITECTURE DEFINITION
# =========================================================
# Defines the Simple Fully Convolutional Network (SFCN) architecture used
# for brain-age estimation (Section III-C.1 of the paper).
#
# Architecture overview:
#   • 5 × ConvBlock layers with progressively increasing channel depth
#     (1→32→64→128→256→256).  Each block contains:
#       - 3-D convolution (3×3×3, no bias)
#       - InstanceNorm3d (not BatchNorm; crucial for small batch sizes < 8)
#       - ReLU activation
#       - Optional MaxPool3d (stride-2 spatial downsampling)
#   • block6: 1×1 conv that bottlenecks from 256 → 64 channels.
#   • AdaptiveAvgPool3d → flattens to a 64-d feature vector.
#   • Dropout (p=0.5 default) for regularisation.
#   • Linear head: 64 → output_dim (number of age bins when used with
#     the soft-label KL-divergence loss in Cell 11).
#
# The in-notebook SFCN defined here is architecturally compatible with the
# pretrained UKBiobank weights loaded later via sfcn_model.load_pretrained_sfcn().
# The forward-pass sanity check (dummy input 1×1×160×192×160) ensures the
# model compiles and runs without shape errors before any training starts.

# =========================================================
# CELL 6 — SFCN ARCHITECTURE (HPC OPTIMIZED)
# =========================================================
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, max_pool=True):
        super().__init__()
        layers = [
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            # VITAL FOR HIGH ACCURACY AT SMALL BATCH SIZES
            nn.InstanceNorm3d(out_channels, affine=True),
            nn.ReLU(inplace=True),
        ]
        if max_pool:
            layers.append(nn.MaxPool3d(kernel_size=2, stride=2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

class SFCN(nn.Module):
    def __init__(self, output_dim=1, dropout_p=0.5):
        super().__init__()
        self.block1 = ConvBlock(1, 32)
        self.block2 = ConvBlock(32, 64)
        self.block3 = ConvBlock(64, 128)
        self.block4 = ConvBlock(128, 256)
        self.block5 = ConvBlock(256, 256)
        self.block6 = nn.Sequential(
            nn.Conv3d(256, 64, kernel_size=1, bias=False),
            nn.InstanceNorm3d(64, affine=True),
            nn.ReLU(inplace=True),
        )
        self.global_pool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.dropout = nn.Dropout(p=dropout_p)
        self.fc = nn.Linear(64, output_dim)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.block6(x)
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        return self.fc(x).squeeze(-1)

model_check = SFCN().to(device)
n_params = sum(p.numel() for p in model_check.parameters())

print("=" * 60)
print("CELL 6: SFCN ARCHITECTURE (HPC STABILIZED)")
print("=" * 60)
print(f"Device          : {device}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
print(f"Parameters      : {n_params:,}")

# Forward-pass sanity check
with torch.no_grad():
    dummy = torch.zeros((1, 1, 160, 192, 160), device=device)
    out = model_check(dummy)

print("Forward pass    : PASS")
del model_check, dummy, out
if torch.cuda.is_available():
    torch.cuda.empty_cache()

CELL 6: SFCN ARCHITECTURE (HPC STABILIZED)
Device          : cuda
GPU             : NVIDIA GeForce RTX 5080
Parameters      : 2,949,601
Forward pass    : PASS


## CELL 7 — Dataset
Defines the dataset loader and tests one real preprocessed sample.


In [3]:
# =========================================================
# CELL 7 — DATASET (RAM-CACHED + AUGMENTATION)
# =========================================================
# Defines `BrainDataset`, the PyTorch Dataset that loads pre-processed
# 3-D brain scans from .npz files (output of preprocessing.ipynb).
#
# Design decisions:
#   cache_in_ram=True (default)
#     Pre-loads all .npz arrays into system RAM on __init__.
#     Rationale: The HPC node has 128 GB RAM; WSL2 I/O from the NTFS volume
#     is slow and occasionally crashes mid-epoch.  Storing tensors as float16
#     keeps peak RAM usage ≈ 33 GB for the full dataset, well within budget.
#     float16 → float32 cast happens lazily in __getitem__ to avoid storing
#     large float32 arrays.
#
#   Augmentation pipeline (training only, via torchio):
#     • RandomAffine(scales=±5%, degrees=±5°, p=0.5)  — mild spatial jitter
#       to improve generalisation across scanner-specific head positioning.
#     • RandomGamma(log_gamma=±0.3, p=0.3)  — intensity/contrast variation,
#       simulating different T1 contrast settings.
#     • RandomNoise(std=0.05, p=0.3)  — Gaussian noise, acts as a regulariser
#       and mimics thermal noise differences between scanner models.
#
# Returns (tensor, age_float32) pairs, where tensor shape is (1, H, W, D)
# with a single grey-matter channel.

# =========================================================
# CELL 7 — DATASET (RAM CACHED + ADVANCED AUGMENTATION)
# =========================================================
import numpy as np
import torch
import torchio as tio
from torch.utils.data import Dataset

# FIX: Swapped to the standard CLI tqdm to bypass the missing ipywidgets module
from tqdm import tqdm  

class BrainDataset(Dataset):
    def __init__(self, df, is_train=False, cache_in_ram=True):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train
        self.cache_in_ram = cache_in_ram
        self.data_cache = []

        # Advanced augmentations to maximize model generalization
        self.transform = tio.Compose([
            tio.RandomAffine(scales=(0.95, 1.05), degrees=5, p=0.5),
            tio.RandomGamma(log_gamma=(-0.3, 0.3), p=0.3), # Helps with contrast variations
            tio.RandomNoise(std=0.05, p=0.3),
        ]) if is_train else None

        # Load safely into your 128GB RAM to bypass WSL I/O crashes
        if self.cache_in_ram:
            print(f"Caching {len(self.df)} scans into RAM (Train={is_train})...")
            for filepath in tqdm(self.df["file_path"], desc="Loading NPZ"):
                with np.load(filepath) as z:
                    # float16 takes ~33GB total for the dataset, easily fitting in 128GB
                    self.data_cache.append(z["data"].astype(np.float16))

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        if self.cache_in_ram:
            data = self.data_cache[idx].astype(np.float32)
        else:
            with np.load(self.df.iloc[idx]["file_path"]) as z:
                data = z["data"].astype(np.float32)

        tensor = torch.from_numpy(data).unsqueeze(0)

        if self.transform:
            tensor = self.transform(tensor)

        age = torch.tensor(float(self.df.iloc[idx]["age"]), dtype=torch.float32)
        return tensor, age

print("=" * 60)
print("CELL 7: DATASET WITH RAM CACHING DEFINED")
print("=" * 60)

/mnt/d/brain age prep and meth/wsl_env_312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CELL 7: DATASET WITH RAM CACHING DEFINED


## CELL 8 — Manifest + metadata
Builds one row per preprocessed scan, joins scan metadata and CDR/MMSE data, and keeps chronological age from CDR when available.


In [4]:
# =========================================================
# CELL 8 — MANIFEST CONSTRUCTION (LINUX PATHS, WSL/HPC)
# =========================================================
# Builds the master `manifest` DataFrame that links every preprocessed .npz
# scan file on disk to its subject, session, run, chronological age, and CDR
# (Clinical Dementia Rating) score.
#
# Steps:
#   1. glob() discovers all *_preproc.npz files under PREPROC_DIR.
#   2. zipfile.is_zipfile() filters out any corrupted or zero-byte .npz files
#      that would otherwise crash np.load() mid-training.
#   3. parse_preproc_filename() extracts (subject_id, day, run, mri_label)
#      from the standardised BIDS-style filename pattern.
#   4. The raw file list is joined to:
#        scan_meta (okarun…csv) — age, MRI-label lookup.
#        cdr (OASIS3_UDSb4_cdr_final.csv) — CDRTOT score per session.
#      Joins are on mri_label (format: OAS<ID>_MR_d<day>).
#
# Paths use /mnt/d/... (Linux mount point) because this cell is designed to
# run either inside WSL2 on the development machine or directly on the HPC
# Linux node.  Adjust PREPROC_DIR / SCAN_META_CSV / CDR_CSV if you switch
# back to a native Windows kernel.
#
# Output: `manifest` — one row per valid scan, columns include
#   subject_id, day, run, mri_label, file_path, age, CDRTOT.

# =========================================================
# CELL 8 — MANIFEST + METADATA (LINUX PATHS)
# =========================================================
import os
import re
import glob
import zipfile
import pandas as pd

# UPDATED TO LINUX PATHS FOR WSL/HPC compatibility
SCAN_META_CSV = "/mnt/d/brain age prep and meth/okarun143_8_17_2026_12_36_37.csv"
CDR_CSV = "/mnt/d/brain age prep and meth/OASIS3_UDSb4_cdr_final.csv"
PREPROC_DIR = "/mnt/d/brain age prep and meth/T1 preprocessed v2"

def parse_preproc_filename(path):
    fname = os.path.basename(path)
    m = re.search(r"sub-(OAS\d+)_sess?-d(\d+)(?:_run-(\d+))?", fname)
    if not m:
        raise ValueError(f"Could not parse subject/session/run from filename: {fname}")

    subject_id, day_str, run = m.group(1), m.group(2), m.group(3)
    run = run or "01"

    return {
        "subject_id": subject_id,
        "day": int(day_str),
        "run": run,
        "mri_label": f"{subject_id}_MR_d{day_str}",
        "file_path": path,
    }

def build_manifest(preproc_dir, scan_meta_csv, cdr_csv):
    files = sorted(
        glob.glob(os.path.join(preproc_dir, "**", "*_preproc.npz"), recursive=True)
    )
    if not files:
        raise FileNotFoundError(f"No preprocessed .npz files found under {preproc_dir}")

    valid_files = [f for f in files if zipfile.is_zipfile(f)]
    corrupted = [f for f in files if not zipfile.is_zipfile(f)]

    if corrupted:
        print(f"Skipping corrupted NPZ files: {len(corrupted)}")
        for f in corrupted[:10]:
            print(" ", f)

    rows = []
    parse_failures = []

    for f in valid_files:
        try:
            rows.append(parse_preproc_filename(f))
        except ValueError as e:
            parse_failures.append(str(e))

    if parse_failures:
        print(f"Filename parsing failures: {len(parse_failures)}")
        for msg in parse_failures[:10]:
            print(" ", msg)

    manifest = pd.DataFrame(rows)

    scan_meta = (
        pd.read_csv(scan_meta_csv)[["Label", "Age", "M/F"]]
        .rename(columns={"Label": "mri_label", "Age": "age_meta", "M/F": "sex"})
    )

    cdr = (
        pd.read_csv(cdr_csv)[["MRI Label", "age at visit", "CDRTOT", "CDRSUM", "MMSE"]]
        .rename(columns={"MRI Label": "mri_label", "age at visit": "age_cdr"})
    )

    # Prevent duplicate metadata columns
    scan_meta = scan_meta.drop_duplicates("mri_label")
    cdr = cdr.drop_duplicates("mri_label")

    manifest = manifest.merge(scan_meta, on="mri_label", how="left")
    manifest = manifest.merge(cdr, on="mri_label", how="left")

    manifest["age"] = pd.to_numeric(
        manifest["age_cdr"], errors="coerce"
    ).fillna(
        pd.to_numeric(manifest["age_meta"], errors="coerce")
    )

    manifest = manifest.drop(columns=["age_meta", "age_cdr"])
    manifest = manifest.dropna(subset=["age"]).reset_index(drop=True)

    return manifest

manifest = build_manifest(PREPROC_DIR, SCAN_META_CSV, CDR_CSV)

print("=" * 60)
print("CELL 8: MANIFEST + METADATA")
print("=" * 60)
print(f"Scans       : {len(manifest)}")
print(f"Subjects    : {manifest['subject_id'].nunique()}")
print(f"Age range   : {manifest['age'].min():.2f} - {manifest['age'].max():.2f}")
print(f"CDR 0.0     : {(manifest['CDRTOT'] == 0.0).sum()}")
print(f"Columns     : {list(manifest.columns)}")

display(manifest.head())

CELL 8: MANIFEST + METADATA
Scans       : 3336
Subjects    : 1098
Age range   : 42.00 - 97.47
CDR 0.0     : 2321
Columns     : ['subject_id', 'day', 'run', 'mri_label', 'file_path', 'sex', 'CDRTOT', 'CDRSUM', 'MMSE', 'age']


,subject_id,day,run,mri_label,file_path,sex,CDRTOT,CDRSUM,MMSE,age
0,OAS30001,129,01,OAS30001_MR_d0129,/mnt/d/brain age prep and meth/T1 preprocessed...,F,0.0,0.0,28.0,65.19
1,OAS30001,129,02,OAS30001_MR_d0129,/mnt/d/brain age prep and meth/T1 preprocessed...,F,0.0,0.0,28.0,65.19
2,OAS30001,757,01,OAS30001_MR_d0757,/mnt/d/brain age prep and meth/T1 preprocessed...,F,0.0,0.0,30.0,67.17
3,OAS30001,757,02,OAS30001_MR_d0757,/mnt/d/brain age prep and meth/T1 preprocessed...,F,0.0,0.0,30.0,67.17
4,OAS30001,2430,01,OAS30001_MR_d2430,/mnt/d/brain age prep and meth/T1 preprocessed...,F,0.0,0.0,30.0,71.17


## CELL 9 — Train / Validation / Test split
Splits at the **subject level**, so scans from the same person cannot appear in multiple splits.


In [5]:
# =========================================================
# CELL 9 — SUBJECT-LEVEL TRAIN / VAL / TEST SPLIT
# =========================================================
# Partitions all subjects (not scans) into three non-overlapping sets.
# Splitting at the *subject* level is mandatory: a subject can have multiple
# scans across different sessions, so a naive random scan-level split would
# allow the same person's data to appear in both train and test, causing
# severe data leakage and optimistically inflated MAE figures.
#
# Split ratios: 70% train / 15% val / 15% test (of subjects).
# The adjusted val_size formula compensates for the sequential two-step
# split (train+test first, then train→train+val) so the final proportions
# match the intended 70/15/15.
#
# Assertion checks confirm zero overlap between all three subject sets.
#
# Post-split filtering:
#   train_cn, val_cn — scans restricted to CDR = 0.0 (cognitively normal).
#     Brain-age models must be trained on healthy brains to learn a reference
#     ageing trajectory; including dementia scans would conflate pathological
#     atrophy with normal ageing.
#   test_df — kept unrestricted (includes CDR > 0) so that BAG / survival
#     analyses in later cells can be run across the full clinical spectrum.


# =========================================================
# CELL 9 — SUBJECT-LEVEL TRAIN / VAL / TEST SPLIT
# =========================================================
from sklearn.model_selection import train_test_split

def subject_level_split(manifest, test_size=0.15, val_size=0.15, random_state=42):
    subjects = manifest["subject_id"].unique()

    train_subj, test_subj = train_test_split(
        subjects, test_size=test_size, random_state=random_state
    )

    train_subj, val_subj = train_test_split(
        train_subj,
        test_size=val_size / (1 - test_size),
        random_state=random_state,
    )

    train_df = manifest[manifest["subject_id"].isin(train_subj)].reset_index(drop=True)
    val_df = manifest[manifest["subject_id"].isin(val_subj)].reset_index(drop=True)
    test_df = manifest[manifest["subject_id"].isin(test_subj)].reset_index(drop=True)

    assert not (set(train_df["subject_id"]) & set(val_df["subject_id"]))
    assert not (set(train_df["subject_id"]) & set(test_df["subject_id"]))
    assert not (set(val_df["subject_id"]) & set(test_df["subject_id"]))

    return train_df, val_df, test_df


train_df, val_df, test_df = subject_level_split(manifest)

# Training/validation are restricted to cognitively normal scans.
train_cn = train_df[train_df["CDRTOT"] == 0.0].reset_index(drop=True)
val_cn = val_df[val_df["CDRTOT"] == 0.0].reset_index(drop=True)

print("=" * 60)
print("CELL 9: TRAIN / VALIDATION / TEST SPLIT")
print("=" * 60)
print(f"Train subjects : {train_df['subject_id'].nunique()} | scans: {len(train_df)}")
print(f"Val subjects   : {val_df['subject_id'].nunique()} | scans: {len(val_df)}")
print(f"Test subjects  : {test_df['subject_id'].nunique()} | scans: {len(test_df)}")
print(f"Train CN       : {len(train_cn)}")
print(f"Val CN         : {len(val_cn)}")
print(f"Test CN        : {(test_df['CDRTOT'] == 0.0).sum()}")

if len(train_cn) == 0 or len(val_cn) == 0:
    raise RuntimeError("No CDR 0.0 scans available for training/validation.")


CELL 9: TRAIN / VALIDATION / TEST SPLIT
Train subjects : 768 | scans: 2361
Val subjects   : 165 | scans: 480
Test subjects  : 165 | scans: 495
Train CN       : 1640
Val CN         : 323
Test CN        : 358


In [ ]:
# =========================================================
# CELL 11 — HETEROGENEOUS ENSEMBLE TRAINING (Section III-C)
# =========================================================
# Trains the 3-model heterogeneous ensemble that forms the core of the
# brain-age pipeline:
#   Model 0 — SFCN pretrained on UKBiobank (seed 0), fine-tuned
#   Model 1 — SFCN pretrained on UKBiobank (seed 1), fine-tuned
#   Model 2 — 3-D ResNet-18 trained from scratch
#
# Key design choices (all vs. original Cell 11):
#
# 1. Pretrained SFCN fine-tuning (differential learning rates)
#    Blocks 1-4 are frozen; only block5 + head are updated (unfreeze_from_block=5).
#    Backbone LR = 1e-5, head LR = 1e-4 — a 10× ratio prevents catastrophic
#    forgetting of the UKBiobank representation while allowing the head to
#    adapt to the OASIS-3 age range.
#
# 2. Gaussian soft-label KL-divergence loss (my_KLDivLoss)
#    Replaces the original nn.L1Loss.  Each integer-year label is spread into
#    a Gaussian over the age-bin vector (sigma=1.0 year) before computing
#    KL-divergence against the model's log-softmax output.  This encodes
#    ordinal age structure into the training signal and improves calibration.
#
# 3. Data-adaptive age-bin range
#    BIN_RANGE is derived from the actual min/max age in train_cn (± 3 years)
#    rather than the UKBiobank's fixed [42, 82] range, ensuring the output
#    probability vector covers the full OASIS-3 age span without wasted bins.
#
# 4. AMP + torch.compile (RTX 5080 / Blackwell)
#    USE_AMP=True enables float16 mixed-precision via torch.autocast, halving
#    VRAM usage and roughly doubling throughput on Blackwell.
#    USE_COMPILE=True wraps each model in torch.compile() for additional XLA-
#    style kernel fusion speedup (Triton backend on CUDA ≥ 12).
#
# 5. OOM guard
#    The inner training loop catches RuntimeError("out of memory"), drains the
#    CUDA cache, and skips the batch rather than crashing the epoch.
#
# 6. Early stopping (patience=8) with ReduceLROnPlateau
#    Checkpoints are saved to CHECKPOINT_DIR whenever val MAE improves;
#    the best weights are reloaded at the end of each model's training.
#
# Outputs: `models`, `model_tags`, `histories` lists — consumed by Cells 12-14.

# =========================================================
# TRAIN ENSEMBLE v2 — Revision 2 (Section III-C)
# =========================================================
"""
Drop-in replacement for your original Cells 11-14 (train ensemble,
validation, test predictions, brain age + BAG). It expects everything
from your existing Cells 6-10 to already exist in the notebook namespace:

    manifest, train_df, val_df, test_df, train_cn, val_cn,
    BrainDataset, NUM_WORKERS, device, RESULTS_DIR, CHECKPOINT_DIR

Everything from Section III-B (systemic age-bias correction, linregress)
and III-A (subject-level split) is UNCHANGED and reused as-is; only the
model / loss / ensemble composition (III-C) changes.

WHAT CHANGED vs. your original Cell 11:
  1. 2 of the 3 ensemble members are pretrained SFCN (UKBiobank_deep_pretrain),
     fine-tuned with differential LR on the final conv block + head only.
  2. The 3rd member is a 3D-ResNet-18 (structurally different architecture,
     trained from scratch — no pretrained 3D-ResNet weights ship in the
     repo you provided; see resnet3d.py docstring).
  3. Loss is Gaussian soft-label + KL divergence over discretized age bins
     (nn.L1Loss() is gone).
  4. Batch sizes are raised to use the RTX 5080's 16GB VRAM headroom
     (SFCN: 8, ResNet3D: 6 — both still AMP fp16).
"""

import os
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from scipy.stats import linregress
from sfcn_model import *
from resnet3d import *
from brainage_utils import *

from sfcn_model import (
    SFCN, load_pretrained_sfcn, freeze_sfcn_backbone,
    differential_lr_param_groups, expected_age_from_logprobs,
)
from resnet3d import resnet18
from brainage_utils import num2vect, my_KLDivLoss

# ---------------------------------------------------------
# Config — hardware-tailored for i9 / 128GB RAM / RTX 5080 16GB
# ---------------------------------------------------------
SFCN_CHECKPOINT = "./UKBiobank_deep_pretrain-master/brain_age/run_20190719_00_epoch_best_mae.p"
BATCH_SIZE_SFCN = 8
BATCH_SIZE_RESNET = 6
NUM_WORKERS = 0
MAX_EPOCHS = 60
PATIENCE = 8
BIN_STEP = 1
SIGMA = 1.0
USE_AMP = torch.cuda.is_available()
USE_COMPILE = torch.cuda.is_available()  # RTX 50-series (Blackwell) supports torch.compile
RESULTS_DIR = "./results_v2" 
CHECKPOINT_DIR = "./checkpoints_v2"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ---------------------------------------------------------
# Age-bin scheme, derived from YOUR data (not UKB's [42,82])
# ---------------------------------------------------------
_age_min = float(np.floor(train_cn["age"].min())) - 3
_age_max = float(np.ceil(train_cn["age"].max())) + 3
if (_age_max - _age_min) % BIN_STEP != 0:
    _age_max += BIN_STEP - ((_age_max - _age_min) % BIN_STEP)  # keep divisible by bin_step
BIN_RANGE = (_age_min, _age_max)
N_BINS = int((BIN_RANGE[1] - BIN_RANGE[0]) / BIN_STEP)
_, _bin_centers_np = num2vect(np.array([BIN_RANGE[0]]), BIN_RANGE, BIN_STEP, SIGMA)
BIN_CENTERS = torch.tensor(_bin_centers_np, dtype=torch.float32)

print("=" * 70)
print("AGE-BIN SCHEME (derived from train_cn, not hardcoded from UK Biobank)")
print("=" * 70)
print(f"bin_range={BIN_RANGE}, bin_step={BIN_STEP}, sigma={SIGMA}, n_bins={N_BINS}")

# ---------------------------------------------------------
# Datasets built ONCE, DataLoaders per-architecture on top of them
# (avoids re-caching the ~33GB volume set twice)
# ---------------------------------------------------------
train_ds = BrainDataset(train_cn, is_train=True, cache_in_ram=True)
val_ds = BrainDataset(val_cn, is_train=False, cache_in_ram=True)
test_ds = BrainDataset(test_df, is_train=False, cache_in_ram=True)


def make_loaders(batch_size):
    kwargs = dict(batch_size=batch_size, num_workers=NUM_WORKERS,
                  pin_memory=torch.cuda.is_available())
    return (
        DataLoader(train_ds, shuffle=True, **kwargs),
        DataLoader(val_ds, shuffle=False, **kwargs),
        DataLoader(test_ds, shuffle=False, **kwargs),
    )


# ---------------------------------------------------------
# Soft-label training / eval epoch (shared by both architectures)
# ---------------------------------------------------------
def run_epoch(model, loader, optimizer, train=True):
    model.train() if train else model.eval()
    total_loss, total_abs_error, n = 0.0, 0.0, 0
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP) if train else None
    context = torch.enable_grad() if train else torch.no_grad()

    with context:
        for inputs, targets in loader:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            y_soft, _ = num2vect(targets.detach().cpu().numpy(), BIN_RANGE, BIN_STEP, SIGMA)
            y_soft = torch.tensor(y_soft, dtype=torch.float32, device=device)

            if train:
                optimizer.zero_grad(set_to_none=True)

            try:
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                    log_probs = model(inputs).reshape(inputs.shape[0], -1)
                    loss = my_KLDivLoss(log_probs, y_soft)

                if train:
                    if USE_AMP:
                        scaler.scale(loss).backward()
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        loss.backward()
                        optimizer.step()

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print("CUDA OOM: skipping batch.")
                    torch.cuda.empty_cache()
                    continue
                raise

            with torch.no_grad():
                pred_age = expected_age_from_logprobs(log_probs.detach().float(), BIN_CENTERS)
                total_abs_error += torch.abs(pred_age - targets).sum().item()

            bs = inputs.size(0)
            total_loss += float(loss.detach().item()) * bs
            n += bs

    if n == 0:
        return float("nan"), float("nan")
    return total_loss / n, total_abs_error / n


def train_one_model(model, optimizer, scheduler, train_loader, val_loader, tag, model_idx):
    best_path = os.path.join(CHECKPOINT_DIR, f"model_{model_idx}_{tag}_best.pt")
    best_val, patience_counter = float("inf"), 0
    history = {"train_mae": [], "val_mae": []}

    print("\n" + "=" * 70)
    print(f"MODEL {model_idx + 1} ({tag})")
    print("=" * 70)

    for epoch in range(MAX_EPOCHS):
        t0 = time.time()
        _, train_mae = run_epoch(model, train_loader, optimizer, train=True)
        _, val_mae = run_epoch(model, val_loader, optimizer, train=False)
        scheduler.step(val_mae)

        history["train_mae"].append(train_mae)
        history["val_mae"].append(val_mae)
        print(f"Epoch {epoch+1:02d}/{MAX_EPOCHS} | Train MAE={train_mae:.2f} | "
              f"Val MAE={val_mae:.2f} | LR={optimizer.param_groups[0]['lr']:.6f} | "
              f"{time.time()-t0:.1f}s")

        if val_mae < best_val:
            best_val, patience_counter = val_mae, 0
            torch.save(model.state_dict(), best_path)
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"Early stopping at epoch {epoch+1} (best Val MAE={best_val:.2f})")
                break
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

    model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))
    print(f"Model {model_idx+1} ({tag}) complete | Best Val MAE={best_val:.3f} years")
    return model, history


def maybe_compile(model):
    if USE_COMPILE:
        try:
            return torch.compile(model)
        except Exception as e:
            print(f"torch.compile unavailable ({e}); continuing uncompiled.")
    return model


# ---------------------------------------------------------
# Build + train the heterogeneous ensemble
# ---------------------------------------------------------
models, model_tags, histories = [], [], {}

sfcn_train_loader, sfcn_val_loader, sfcn_test_loader = make_loaders(BATCH_SIZE_SFCN)

for seed in (0, 1):
    torch.manual_seed(seed)
    np.random.seed(seed)
    sfcn = load_pretrained_sfcn(SFCN_CHECKPOINT, n_bins=N_BINS, device=device)
    sfcn = freeze_sfcn_backbone(sfcn, unfreeze_from_block=5)
    sfcn = maybe_compile(sfcn)
    optimizer = torch.optim.AdamW(differential_lr_param_groups(sfcn, backbone_lr=1e-5, head_lr=1e-4),
                                   weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-7)
    sfcn, hist = train_one_model(sfcn, optimizer, scheduler, sfcn_train_loader, sfcn_val_loader,
                                  tag="sfcn_pretrained", model_idx=len(models))
    models.append(sfcn); model_tags.append("sfcn_pretrained"); histories[len(models)-1] = hist

resnet_train_loader, resnet_val_loader, resnet_test_loader = make_loaders(BATCH_SIZE_RESNET)

torch.manual_seed(2)
np.random.seed(2)
resnet = resnet18(num_classes=N_BINS, dropout=True).to(device)
resnet = maybe_compile(resnet)
optimizer = torch.optim.AdamW(resnet.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-7)
resnet, hist = train_one_model(resnet, optimizer, scheduler, resnet_train_loader, resnet_val_loader,
                                tag="resnet18_scratch", model_idx=len(models))
models.append(resnet); model_tags.append("resnet18_scratch"); histories[len(models)-1] = hist

print("\n" + "=" * 70)
print("ALL MODELS TRAINED")
for i, tag in enumerate(model_tags):
    print(f"Model {i+1} ({tag}) best Val MAE: {min(histories[i]['val_mae']):.3f} years")

# ---------------------------------------------------------
# Ensemble prediction helper (handles each model's own preferred batch size)
# ---------------------------------------------------------
_BATCH_BY_TAG = {"sfcn_pretrained": BATCH_SIZE_SFCN, "resnet18_scratch": BATCH_SIZE_RESNET}


@torch.no_grad()
def ensemble_predict(models, tags, df):
    all_model_preds = []
    for model, tag in zip(models, tags):
        model.eval()
        loader = DataLoader(BrainDataset(df, is_train=False), batch_size=_BATCH_BY_TAG[tag],
                             shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
        preds = []
        for inputs, _ in loader:
            inputs = inputs.to(device, non_blocking=True)
            log_probs = model(inputs)
            pred_age = expected_age_from_logprobs(log_probs, BIN_CENTERS)
            preds.extend(pred_age.detach().cpu().numpy().tolist())
        all_model_preds.append(np.asarray(preds, dtype=float))

    stacked = np.vstack(all_model_preds)
    out = df.copy()
    out["predicted_age"] = stacked.mean(axis=0)
    out["predicted_age_sd"] = stacked.std(axis=0)
    return out, stacked



In [ ]:
# =========================================================
# CELL 12 (CV DIAGNOSTIC) — 5-FOLD CROSS-VALIDATION
# =========================================================
# Secondary diagnostic cell — does NOT replace the production 70/15/15 split
# and does NOT touch test_df.  Its sole purpose is to put a confidence
# interval on the MAE that the model would achieve on a genuinely unseen fold,
# because a single-split estimate from ≈ 300-400 subjects has high variance.
#
# Protocol:
#   • GroupKFold(n_splits=5) on the pooled (train + val) CDR-0 scans.
#     Groups = subject_id → no subject leaks across folds.
#   • One pretrained SFCN is fine-tuned per fold (not the full 3-model
#     ensemble) to keep the 5× compute cost manageable.  The SFCN members
#     dominate ensemble accuracy, so this is a representative proxy.
#   • MAX_EPOCHS=30, PATIENCE=5 — backbone is already pretrained, so
#     convergence is faster than training from scratch.
#
# Outputs printed at the end:
#   Per-fold MAE list, mean ± 95% CI.
# Compare the CI against the production test MAE (Cell 17) to judge whether
# that point estimate falls within the natural fold-to-fold spread.

# =========================================================
# CV DIAGNOSTIC — Section III-A robustness addition
# =========================================================
"""
Standalone, SECONDARY diagnostic. It does NOT replace your production
train/val/test split (train_ensemble_v2.py) or touch test_df at all — it
only exists to put a confidence interval on the MAE you'd expect to see on
an unseen fold, since a single 70/15/15 split gives one point estimate that
is fragile with only a few hundred subjects.

Uses ONE pretrained-and-fine-tuned SFCN per fold (not the full 3-model
heterogeneous ensemble) to keep 5x the training cost reasonable — the
production ensemble's accuracy is dominated by the pretrained-SFCN
members anyway, so this is a fast, honest proxy for expected generalization.

Requires: manifest, train_df, val_df, test_df, BrainDataset, device,
NUM_WORKERS, CHECKPOINT_DIR already defined (same as train_ensemble_v2.py).
"""

import os
import numpy as np
import torch
from sklearn.model_selection import GroupKFold
from torch.utils.data import DataLoader

from sfcn_model import load_pretrained_sfcn, freeze_sfcn_backbone, differential_lr_param_groups, expected_age_from_logprobs
from brainage_utils import num2vect, my_KLDivLoss

SFCN_CHECKPOINT = "./UKBiobank_deep_pretrain-master/brain_age/run_20190719_00_epoch_best_mae.p"
K_FOLDS = 5
BATCH_SIZE = 8
MAX_EPOCHS = 30       # fine-tuning a mostly-frozen backbone converges faster than from-scratch training
PATIENCE = 5
BIN_STEP, SIGMA = 1, 1.0
USE_AMP = torch.cuda.is_available()

# Pool everything EXCEPT the held-out test set's subjects.
cv_pool = pd_concat = __import__("pandas").concat([train_df, val_df], ignore_index=True)
cv_pool_cn = cv_pool[cv_pool["CDRTOT"] == 0.0].reset_index(drop=True)

gkf = GroupKFold(n_splits=K_FOLDS)
fold_maes = []

for fold_idx, (tr_idx, va_idx) in enumerate(gkf.split(cv_pool_cn, groups=cv_pool_cn["subject_id"])):
    fold_train = cv_pool_cn.iloc[tr_idx].reset_index(drop=True)
    fold_val = cv_pool_cn.iloc[va_idx].reset_index(drop=True)

    age_min = float(np.floor(fold_train["age"].min())) - 3
    age_max = float(np.ceil(fold_train["age"].max())) + 3
    bin_range = (age_min, age_max)
    n_bins = int((age_max - age_min) / BIN_STEP)
    _, bc_np = num2vect(np.array([age_min]), bin_range, BIN_STEP, SIGMA)
    bin_centers = torch.tensor(bc_np, dtype=torch.float32)

    train_loader = DataLoader(BrainDataset(fold_train, is_train=True, cache_in_ram=True),
                               batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                               pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(BrainDataset(fold_val, is_train=False, cache_in_ram=True),
                             batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                             pin_memory=torch.cuda.is_available())

    torch.manual_seed(fold_idx)
    model = load_pretrained_sfcn(SFCN_CHECKPOINT, n_bins=n_bins, device=device)
    model = freeze_sfcn_backbone(model, unfreeze_from_block=5)
    optimizer = torch.optim.AdamW(differential_lr_param_groups(model, backbone_lr=1e-5, head_lr=1e-4), weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

    best_val_mae, patience_counter = float("inf"), 0
    for epoch in range(MAX_EPOCHS):
        model.train()
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            y_soft, _ = num2vect(targets.cpu().numpy(), bin_range, BIN_STEP, SIGMA)
            y_soft = torch.tensor(y_soft, dtype=torch.float32, device=device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                log_probs = model(inputs).reshape(inputs.shape[0], -1)
                loss = my_KLDivLoss(log_probs, y_soft)
            loss.backward()
            optimizer.step()

        model.eval()
        abs_err, n = 0.0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                log_probs = model(inputs).reshape(inputs.shape[0], -1)
                pred_age = expected_age_from_logprobs(log_probs, bin_centers)
                abs_err += torch.abs(pred_age - targets).sum().item()
                n += targets.shape[0]
        val_mae = abs_err / n
        scheduler.step(val_mae)

        if val_mae < best_val_mae:
            best_val_mae, patience_counter = val_mae, 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break

    print(f"Fold {fold_idx+1}/{K_FOLDS}: best Val MAE = {best_val_mae:.3f} years "
          f"(n_train={len(fold_train)}, n_val={len(fold_val)})")
    fold_maes.append(best_val_mae)

fold_maes = np.array(fold_maes)
mean_mae = fold_maes.mean()
sem_mae = fold_maes.std(ddof=1) / np.sqrt(K_FOLDS)
ci95 = 1.96 * sem_mae

print("\n" + "=" * 60)
print("5-FOLD SUBJECT-LEVEL CV — MAE ROBUSTNESS CHECK")
print("=" * 60)
print(f"Per-fold MAE: {np.round(fold_maes, 3).tolist()}")
print(f"Mean MAE: {mean_mae:.3f} years  (95% CI: {mean_mae-ci95:.3f} - {mean_mae+ci95:.3f})")
print("Compare this range against your single-split test MAE from train_ensemble_v2.py "
      "to see whether that point estimate is inside or outside the fold-to-fold spread.")


In [6]:
# =========================================================
# V2 ENSEMBLE RECOVERY & CONFIGURATION
# =========================================================
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
from sfcn_model import load_pretrained_sfcn, expected_age_from_logprobs
from resnet3d import resnet18
from brainage_utils import num2vect

SFCN_CHECKPOINT = "./UKBiobank_deep_pretrain-master/brain_age/run_20190719_00_epoch_best_mae.p"
BATCH_SIZE_SFCN = 8
BATCH_SIZE_RESNET = 6
NUM_WORKERS = 0
RESULTS_DIR = "./results_v2" 
CHECKPOINT_DIR = "./checkpoints_v2"
BIN_STEP, SIGMA = 1, 1.0

_age_min = float(np.floor(train_cn["age"].min())) - 3
_age_max = float(np.ceil(train_cn["age"].max())) + 3
if (_age_max - _age_min) % BIN_STEP != 0:
    _age_max += BIN_STEP - ((_age_max - _age_min) % BIN_STEP)
BIN_RANGE = (_age_min, _age_max)
N_BINS = int((BIN_RANGE[1] - BIN_RANGE[0]) / BIN_STEP)
_, _bin_centers_np = num2vect(np.array([BIN_RANGE[0]]), BIN_RANGE, BIN_STEP, SIGMA)
BIN_CENTERS = torch.tensor(_bin_centers_np, dtype=torch.float32)

print("Reloading heterogeneous v2 ensemble from checkpoints...")
models = []
model_tags = ["sfcn_pretrained", "sfcn_pretrained", "resnet18_scratch"]

def load_compiled_weights(model, path):
    state_dict = torch.load(path, map_location=device, weights_only=True)
    clean_state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(clean_state_dict)
    model.eval()
    return model

m1 = load_pretrained_sfcn(SFCN_CHECKPOINT, n_bins=N_BINS, device=device)
m1 = load_compiled_weights(m1, os.path.join(CHECKPOINT_DIR, "model_0_sfcn_pretrained_best.pt"))
models.append(m1)

m2 = load_pretrained_sfcn(SFCN_CHECKPOINT, n_bins=N_BINS, device=device)
m2 = load_compiled_weights(m2, os.path.join(CHECKPOINT_DIR, "model_1_sfcn_pretrained_best.pt"))
models.append(m2)

m3 = resnet18(num_classes=N_BINS, dropout=True).to(device)
m3 = load_compiled_weights(m3, os.path.join(CHECKPOINT_DIR, "model_2_resnet18_scratch_best.pt"))
models.append(m3)

_BATCH_BY_TAG = {"sfcn_pretrained": BATCH_SIZE_SFCN, "resnet18_scratch": BATCH_SIZE_RESNET}

@torch.no_grad()
def ensemble_predict(models, tags, df):
    all_model_preds = []
    for model, tag in zip(models, tags):
        model.eval()
        loader = DataLoader(BrainDataset(df, is_train=False), batch_size=_BATCH_BY_TAG[tag],
                             shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
        preds = []
        for inputs, _ in loader:
            inputs = inputs.to(device, non_blocking=True)
            log_probs = model(inputs)
            pred_age = expected_age_from_logprobs(log_probs, BIN_CENTERS)
            preds.extend(pred_age.detach().cpu().numpy().tolist())
        all_model_preds.append(np.asarray(preds, dtype=float))

    stacked = np.vstack(all_model_preds)
    out = df.copy()
    out["predicted_age"] = stacked.mean(axis=0)
    out["predicted_age_sd"] = stacked.std(axis=0)
    return out, stacked

print("Models and configurations successfully loaded into memory.")

Reloading heterogeneous v2 ensemble from checkpoints...
[load_pretrained_sfcn] loaded 42 tensors, skipped 2 (expected: the final classifier conv, shape mismatch due to n_bins=61)
  skipped: ['classifier.conv_6.weight', 'classifier.conv_6.bias']
[load_pretrained_sfcn] loaded 42 tensors, skipped 2 (expected: the final classifier conv, shape mismatch due to n_bins=61)
  skipped: ['classifier.conv_6.weight', 'classifier.conv_6.bias']
Models and configurations successfully loaded into memory.


## CELL 12 — Validation results
Evaluates each trained model and the ensemble on the validation set.


In [7]:
# =========================================================
# CELL 12 (v2) — VALIDATION RESULTS
# =========================================================
# Evaluates each trained ensemble member and the mean ensemble independently
# on the held-out validation set (val_cn — CDR = 0.0 scans only).
#
# predict_df() helper:
#   • Wraps each model in torch.no_grad() for memory efficiency.
#   • Uses architecture-specific batch sizes (_BATCH_BY_TAG) to avoid OOM.
#   • Calls expected_age_from_logprobs() to convert log-probability vectors
#     over age bins into scalar age predictions via the inner-product with
#     BIN_CENTERS.
#
# Ensemble prediction = simple mean of the three model outputs.
# Printed metrics:
#   • Per-model MAE and RMSE — lets us check that no single member is
#     significantly worse than the others (which would suggest a training
#     failure for that seed/architecture).
#   • Ensemble MAE and RMSE — the figures cited in the paper's Section III-C.4.

# =========================================================
# CELL 12 (v2 COMPATIBLE) — VALIDATION RESULTS
# =========================================================
import numpy as np
import torch
from torch.utils.data import DataLoader

@torch.no_grad()
def predict_df(model, tag, df):
    loader = DataLoader(
        BrainDataset(df, is_train=False),
        batch_size=_BATCH_BY_TAG[tag],
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )
    preds, actual = [], []
    model.eval()
    for inputs, targets in loader:
        inputs = inputs.to(device, non_blocking=True)
        log_probs = model(inputs)
        p = expected_age_from_logprobs(log_probs, BIN_CENTERS).detach().cpu().numpy()
        preds.extend(p.tolist())
        actual.extend(targets.numpy().tolist())
    return np.asarray(actual, dtype=float), np.asarray(preds, dtype=float)

val_model_preds = []
for i, (model, tag) in enumerate(zip(models, model_tags)):
    y_val, p_val = predict_df(model, tag, val_cn)
    val_model_preds.append(p_val)
    mae = float(np.mean(np.abs(p_val - y_val)))
    rmse = float(np.sqrt(np.mean((p_val - y_val) ** 2)))
    print(f"Model {i + 1} ({tag}): MAE={mae:.3f} | RMSE={rmse:.3f}")

val_model_preds = np.vstack(val_model_preds)
val_ensemble_pred = val_model_preds.mean(axis=0)

val_mae = float(np.mean(np.abs(val_ensemble_pred - y_val)))
val_rmse = float(np.sqrt(np.mean((val_ensemble_pred - y_val) ** 2)))
print("-" * 60)
print(f"Ensemble validation MAE : {val_mae:.3f} years")
print(f"Ensemble validation RMSE: {val_rmse:.3f} years")

Caching 323 scans into RAM (Train=False)...


Loading NPZ: 100%|██████████| 323/323 [00:24<00:00, 13.27it/s]


Model 1 (sfcn_pretrained): MAE=4.330 | RMSE=5.906
Caching 323 scans into RAM (Train=False)...


Loading NPZ: 100%|██████████| 323/323 [00:23<00:00, 13.91it/s]


Model 2 (sfcn_pretrained): MAE=4.369 | RMSE=5.974
Caching 323 scans into RAM (Train=False)...


Loading NPZ: 100%|██████████| 323/323 [00:23<00:00, 14.03it/s]


Model 3 (resnet18_scratch): MAE=4.575 | RMSE=6.020
------------------------------------------------------------
Ensemble validation MAE : 3.996 years
Ensemble validation RMSE: 5.377 years


## CELL 13 — Test predictions
Generates ensemble predictions for the held-out test subjects. No test labels are used to train the model or fit the bias correction.


In [8]:
# =========================================================
# CELL 13 (v2) — TEST SET PREDICTIONS
# =========================================================
# Runs the trained heterogeneous ensemble over the full held-out test set
# (test_df — all CDR levels, not restricted to CN) and stores the raw
# predictions before any bias correction.
#
# ensemble_predict() (defined in Cell 11):
#   • Calls each model on the test DataLoader with its architecture-matched
#     batch size.
#   • Stacks per-model predictions, computes mean (point estimate) and
#     standard deviation (ensemble disagreement) per scan.
#   • Returns a DataFrame with columns:
#       subject_id, mri_label, run, age, CDRTOT,
#       predicted_age (ensemble mean),
#       predicted_age_sd (ensemble std — used in Cell 16 noise analysis).
#
# The raw predictions CSV is written to results_v2/ for reproducibility;
# the bias-corrected `brain_age` and `BAG` columns are added in Cell 14.

# =========================================================
# CELL 13 (v2 COMPATIBLE) — TEST PREDICTIONS
# =========================================================
# Uses the ensemble_predict function defined in the training cell
test_predictions, test_stacked = ensemble_predict(models, model_tags, test_df)

test_predictions.to_csv(os.path.join(RESULTS_DIR, "test_predictions_raw_v2.csv"), index=False)

print("=" * 60)
print("CELL 13: TEST PREDICTIONS")
print("=" * 60)
print(f"Test scans       : {len(test_predictions)}")
print(f"Models in ensemble: {len(models)}")
print(f"Prediction SD mean: {test_predictions['predicted_age_sd'].mean():.3f}")
display(test_predictions.head())

Caching 495 scans into RAM (Train=False)...


Loading NPZ: 100%|██████████| 495/495 [00:37<00:00, 13.29it/s]


Caching 495 scans into RAM (Train=False)...


Loading NPZ: 100%|██████████| 495/495 [00:38<00:00, 13.01it/s]


Caching 495 scans into RAM (Train=False)...


Loading NPZ: 100%|██████████| 495/495 [00:37<00:00, 13.25it/s]


CELL 13: TEST PREDICTIONS
Test scans       : 495
Models in ensemble: 3
Prediction SD mean: 1.804


,subject_id,day,run,mri_label,file_path,sex,CDRTOT,CDRSUM,MMSE,age,predicted_age,predicted_age_sd
0,OAS30011,55,01,OAS30011_MR_d0055,/mnt/d/brain age prep and meth/T1 preprocessed...,F,0.0,0.0,28.0,78.52,78.341705,0.239153
1,OAS30011,1671,01,OAS30011_MR_d1671,/mnt/d/brain age prep and meth/T1 preprocessed...,F,0.0,0.0,28.0,82.73,79.740865,1.156252
2,OAS30025,210,01,OAS30025_MR_d0210,/mnt/d/brain age prep and meth/T1 preprocessed...,F,0.0,0.0,26.0,64.86,66.663709,0.750415
3,OAS30025,210,02,OAS30025_MR_d0210,/mnt/d/brain age prep and meth/T1 preprocessed...,F,0.0,0.0,26.0,64.86,66.903776,0.221239
4,OAS30025,2298,01,OAS30025_MR_d2298,/mnt/d/brain age prep and meth/T1 preprocessed...,F,0.0,0.0,26.0,71.01,71.055967,2.540018


## CELL 14 — Brain Age + BAG
Fits the age-prediction bias correction on **training CDR 0.0 scans only**, then applies it to the held-out test predictions.


In [9]:
# =========================================================
# CELL 14 (v2) — SYSTEMIC AGE-BIAS CORRECTION + BAG
# =========================================================
# Applies the standard linear-regression correction for the well-known
# age-related bias in brain-age models (Section III-B of the paper).
#
# Why the bias exists:
#   SFCN (and most regression models) exhibit regression-to-the-mean: younger
#   subjects are predicted as older, older subjects as younger.  If left
#   uncorrected, BAG correlates with chronological age, confounding any
#   downstream group comparison.
#
# Correction procedure:
#   1. Run the *trained* ensemble over the TRAINING set (train_cn) to obtain
#      predicted_age ŷ for each training scan.
#   2. Fit  ŷ = slope × age + intercept  via ordinary least-squares.
#   3. Correct test predictions:
#        brain_age = (predicted_age_test − intercept) / slope
#      This maps the systematic shift back onto the identity line so that
#      E[brain_age | age] ≈ age for healthy controls.
#   4. BAG (Brain Age Gap) = brain_age − chronological_age.
#      BAG ≈ 0 for a typical healthy brain; BAG > 0 suggests accelerated
#      ageing; BAG < 0 suggests preserved / slowed ageing.
#
# Results are saved to brainage_predictions.csv for use in Cells 15-19.

# =========================================================
# CELL 14 (v2 COMPATIBLE) — BRAIN AGE + BAG
# =========================================================
from scipy.stats import linregress
import numpy as np
import pandas as pd

# Generate training predictions for fitting the correction.
# FIXED: Added the missing 'model_tags' argument
train_predictions, _ = ensemble_predict(models, model_tags, train_cn)

x = train_predictions["age"].to_numpy(dtype=float)
y = train_predictions["predicted_age"].to_numpy(dtype=float)

slope, intercept, r_value, p_value, std_err = linregress(x, y)

if abs(slope) < 1e-12:
    raise RuntimeError("Bias-correction slope is too close to zero.")

test_predictions["brain_age"] = (
    test_predictions["predicted_age"] - intercept
) / slope

test_predictions["BAG"] = (
    test_predictions["brain_age"] - test_predictions["age"]
)

test_predictions.to_csv(
    os.path.join(RESULTS_DIR, "brainage_predictions.csv"),
    index=False
)

print("=" * 60)
print("CELL 14: BRAIN AGE + BAG")
print("=" * 60)
print(f"Bias correction slope  : {float(slope):.4f}")
print(f"Bias correction intercept: {float(intercept):.4f}")
print(f"Training r             : {float(r_value):.4f}")
print(f"Training n             : {len(train_predictions)}")
print(f"Mean test BAG          : {test_predictions['BAG'].mean():.3f} years")
print(f"SD test BAG            : {test_predictions['BAG'].std(ddof=1):.3f} years")
display(test_predictions[["subject_id", "mri_label", "age", "predicted_age", "brain_age", "BAG"]].head())

Caching 1640 scans into RAM (Train=False)...


Loading NPZ: 100%|██████████| 1640/1640 [02:03<00:00, 13.25it/s]


Caching 1640 scans into RAM (Train=False)...


Loading NPZ: 100%|██████████| 1640/1640 [02:06<00:00, 13.00it/s]


Caching 1640 scans into RAM (Train=False)...


Loading NPZ: 100%|██████████| 1640/1640 [02:02<00:00, 13.43it/s]


CELL 14: BRAIN AGE + BAG
Bias correction slope  : 0.7463
Bias correction intercept: 17.2145
Training r             : 0.9344
Training n             : 1640
Mean test BAG          : -0.117 years
SD test BAG            : 6.628 years


,subject_id,mri_label,age,predicted_age,brain_age,BAG
0,OAS30011,OAS30011_MR_d0055,78.52,78.341705,81.904953,3.384953
1,OAS30011,OAS30011_MR_d1671,82.73,79.740865,83.779702,1.049702
2,OAS30025,OAS30025_MR_d0210,64.86,66.663709,66.257481,1.397481
3,OAS30025,OAS30025_MR_d0210,64.86,66.903776,66.579149,1.719149
4,OAS30025,OAS30025_MR_d2298,71.01,71.055967,72.142714,1.132714


## CELL 15 — Confusion matrix
Because SFCN is a regression model, this is a **derived age-group classification analysis**, not the model's training objective.

Age groups:
- Young: `< 60`
- Middle-aged: `60–74`
- Older: `>= 75`


In [10]:
# =========================================================
# CELL 15 — DERIVED AGE-GROUP CLASSIFICATION
# =========================================================
# Converts the continuous brain-age regression output into a derived
# three-class classification problem as a supplementary diagnostic view.
# This is NOT part of the model's training objective — it simply illustrates
# how well the bias-corrected brain_age ranks subjects into age groups.
#
# Age bins used:
#   Young    : brain_age < 60
#   Middle   : 60 ≤ brain_age < 75
#   Older    : brain_age ≥ 75
#
# Metrics reported:
#   • Confusion matrix (true vs. predicted group)
#   • Overall accuracy
#   • Per-class Precision / Recall / F1
#   • Weighted Precision, Recall, and macro-F1
#
# The 71.3% accuracy cited in the paper comes from this cell.
# Note: zero_division=0 suppresses spurious warnings for classes with no
# predicted samples (can occur if the model never predicts "Older" for a
# small test set).

# =========================================================
# CELL 15 — DERIVED AGE-GROUP CONFUSION MATRIX
# =========================================================
import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
)

AGE_BINS = [-np.inf, 60, 75, np.inf]
AGE_LABELS = ["Young (<60)", "Middle (60-74)", "Older (>=75)"]

cm_df = test_predictions.copy()

cm_df["true_group"] = pd.cut(
    cm_df["age"],
    bins=AGE_BINS,
    labels=AGE_LABELS,
    right=False,
)

cm_df["pred_group"] = pd.cut(
    cm_df["brain_age"],
    bins=AGE_BINS,
    labels=AGE_LABELS,
    right=False,
)

valid = cm_df.dropna(subset=["true_group", "pred_group"]).copy()

y_true_group = valid["true_group"].astype(str)
y_pred_group = valid["pred_group"].astype(str)

cm = confusion_matrix(
    y_true_group,
    y_pred_group,
    labels=AGE_LABELS
)

cm_table = pd.DataFrame(
    cm,
    index=[f"Actual: {x}" for x in AGE_LABELS],
    columns=[f"Predicted: {x}" for x in AGE_LABELS],
)

accuracy = float(accuracy_score(y_true_group, y_pred_group))

# Fixed typo from original logic where 'recall_recall_fscore_support' was called
precision, recall, f1, support = precision_recall_fscore_support(
    y_true_group,
    y_pred_group,
    labels=AGE_LABELS,
    zero_division=0,
)

print("=" * 60)
print("CELL 15: DERIVED AGE-GROUP CONFUSION MATRIX")
print("=" * 60)
display(cm_table)

print(f"Derived classification accuracy : {accuracy:.3f}")
print(f"Weighted precision              : {precision_recall_fscore_support(y_true_group, y_pred_group, labels=AGE_LABELS, average='weighted', zero_division=0)[0]:.3f}")
print(f"Weighted recall                 : {precision_recall_fscore_support(y_true_group, y_pred_group, labels=AGE_LABELS, average='weighted', zero_division=0)[1]:.3f}")
print(f"Weighted F1                     : {f1.mean():.3f}  [macro F1]")

for label, p, r, f, s in zip(AGE_LABELS, precision, recall, f1, support):
    print(f"{label:16s} | Precision={p:.3f} Recall={r:.3f} F1={f:.3f} Support={s}")

CELL 15: DERIVED AGE-GROUP CONFUSION MATRIX


,Predicted: Young (<60),Predicted: Middle (60-74),Predicted: Older (>=75)
Actual: Young (<60),42,14,0
Actual: Middle (60-74),28,162,52
Actual: Older (>=75),0,50,147


Derived classification accuracy : 0.709
Weighted precision              : 0.712
Weighted recall                 : 0.709
Weighted F1                     : 0.700  [macro F1]
Young (<60)      | Precision=0.600 Recall=0.750 F1=0.667 Support=56
Middle (60-74)   | Precision=0.717 Recall=0.669 F1=0.692 Support=242
Older (>=75)     | Precision=0.739 Recall=0.746 F1=0.742 Support=197


## CELL 16 — Measurement noise
Uses same-day repeat scans with the same `mri_label` and computes the repeat-difference noise floor:
`σ_SEM = SD(differences) / √2`
and
`MDC95 = 1.96 × √2 × σ_SEM`.


In [11]:
# =========================================================
# CELL 16 — SAME-DAY REPEAT MEASUREMENT NOISE FLOOR
# =========================================================
# Quantifies the intra-session measurement noise in the brain-age estimates
# using pairs of same-day repeat scans (Section III-E.2 of the paper).
#
# OASIS-3 contains subjects scanned twice in the same session with a short
# inter-scan interval (repositioning between runs).  Any BAG difference
# between Run 1 and Run 2 is therefore attributable purely to:
#   • Scanner thermal/electronic noise
#   • Head repositioning (residual after MNI registration)
#   • Preprocessing stochasticity (none — the pipeline is deterministic)
#
# Derived statistics:
#   SD_diff = std(BAG_run1 − BAG_run2)  [across all repeat pairs]
#   σ_SEM   = SD_diff / √2
#     — the standard error of a single BAG measurement.
#     — used in Cell 18 to threshold "reliable" longitudinal BAG changes.
#   MDC_95  = 1.96 × √2 × σ_SEM
#     — Minimal Detectable Change at 95% confidence.
#     — A longitudinal BAG change must exceed MDC_95 to be considered a
#       true biological change rather than measurement noise.
#
# Results are also saved to repeat_pairs.csv for reproducibility.

# =========================================================
# CELL 16 — SAME-DAY REPEAT MEASUREMENT NOISE
# =========================================================
import numpy as np
import pandas as pd

def build_repeat_pairs(pred_df):
    rows = []

    for mri_label, group in pred_df.groupby("mri_label"):
        group = group.sort_values("run")

        if len(group) < 2:
            continue

        r1, r2 = group.iloc[0], group.iloc[1]

        rows.append({
            "subject_id": r1["subject_id"],
            "mri_label": mri_label,
            "BAG_run1": float(r1["BAG"]),
            "BAG_run2": float(r2["BAG"]),
            "CDRTOT": r1["CDRTOT"],
        })

    return pd.DataFrame(rows)


# For the noise analysis, use the held-out test predictions.
repeat_pairs = build_repeat_pairs(test_predictions)

print("=" * 60)
print("CELL 16: MEASUREMENT NOISE")
print("=" * 60)
print(f"Same-day repeat pairs: {len(repeat_pairs)}")

if len(repeat_pairs) >= 2:
    differences = (
        repeat_pairs["BAG_run1"].to_numpy(dtype=float)
        - repeat_pairs["BAG_run2"].to_numpy(dtype=float)
    )

    sd_difference = float(np.std(differences, ddof=1))
    sigma_sem = float(sd_difference / np.sqrt(2))
    mdc95 = float(1.96 * np.sqrt(2) * sigma_sem)

    print(f"SD of repeat differences : {sd_difference:.3f} years")
    print(f"Sigma_SEM                : {sigma_sem:.3f} years")
    print(f"MDC (95%)                : {mdc95:.3f} years")

    repeat_pairs.to_csv(
        os.path.join(RESULTS_DIR, "repeat_pairs.csv"),
        index=False
    )

else:
    sigma_sem = np.nan
    mdc95 = np.nan
    print("Not enough repeat pairs to estimate a stable noise floor.")

CELL 16: MEASUREMENT NOISE
Same-day repeat pairs: 149
SD of repeat differences : 2.237 years
Sigma_SEM                : 1.582 years
MDC (95%)                : 4.384 years


## CELL 17 — Final statistics
Prints the main regression, BAG, ensemble, derived classification, and measurement-noise results in one place.


In [12]:
# =========================================================
# CELL 17 — FINAL CONSOLIDATED STATISTICS
# =========================================================
# Aggregates and prints all primary results in one place and saves a
# machine-readable JSON summary to results_v2/final_statistics.json.
#
# Metrics reported (on test set):
#
#   Primary regression (CDR = 0.0 scans only):
#     MAE, RMSE, Pearson r, R²  — measures of how well brain_age tracks
#     chronological age on cognitively normal subjects.  The CDR-0.0 subset
#     is used as the reference because including dementia scans would
#     artificially inflate MAE (brain_age is expected to exceed chrono age
#     for dementia patients).
#
#   Brain Age Gap:
#     Mean BAG, BAG SD  — should be near-zero for a well-calibrated model
#     after the bias correction in Cell 14.
#
#   Derived classification:
#     Accuracy — from Cell 15's three-class confusion matrix.
#
#   Measurement noise (if repeat pairs were available in Cell 16):
#     σ_SEM, MDC_95.
#
# The JSON file is the authoritative record of results used to populate
# Table I and Section IV of the paper.

# =========================================================
# CELL 17 (v2 COMPATIBLE) — FINAL STATISTICS
# =========================================================
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
import json

test_cn_pred = test_predictions[test_predictions["CDRTOT"] == 0.0].copy()

if len(test_cn_pred) > 0:
    y_true = test_cn_pred["age"].to_numpy(dtype=float)
    y_pred = test_cn_pred["brain_age"].to_numpy(dtype=float)
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    pearson_r = float(pearsonr(y_true, y_pred)[0]) if len(y_true) >= 2 else np.nan
else:
    mae = rmse = r2 = pearson_r = np.nan

raw_mae = float(mean_absolute_error(test_predictions["age"], test_predictions["predicted_age"]))
bag_mean = float(test_predictions["BAG"].mean())
bag_sd = float(test_predictions["BAG"].std(ddof=1))
ensemble_sd_mean = float(test_predictions["predicted_age_sd"].mean())

print("\n" + "=" * 70)
print("CELL 17: FINAL BRAIN-AGE RESULTS")
print("=" * 70)

print("\nDATA")
print(f"Total scans in manifest       : {len(manifest)}")
print(f"Train / Val / Test scans      : {len(train_df)} / {len(val_df)} / {len(test_df)}")

print("\nMODEL")
print(f"Models trained                : {len(models)} ({', '.join(model_tags)})")
print(f"Batch size (SFCN / ResNet3D)  : {BATCH_SIZE_SFCN} / {BATCH_SIZE_RESNET}")

print("\nPRIMARY TEST REGRESSION — CDR 0.0")
print(f"MAE                           : {mae:.3f} years")
print(f"RMSE                          : {rmse:.3f} years")
print(f"Pearson r                     : {pearson_r:.3f}")
print(f"R²                            : {r2:.3f}")

print("\nBRAIN AGE / BAG — TEST")
print(f"Mean BAG                      : {bag_mean:.3f} years")
print(f"BAG SD                        : {bag_sd:.3f} years")

print("\nDERIVED AGE-GROUP CLASSIFICATION")
print(f"Accuracy                      : {accuracy:.3f}")

if np.isfinite(sigma_sem):
    print(f"\nMEASUREMENT NOISE\nSigma_SEM: {sigma_sem:.3f} years | MDC (95%): {mdc95:.3f} years")

summary = {
    "mae_years": mae, "rmse_years": rmse, "pearson_r": pearson_r, "r2": r2,
    "raw_mae_years": raw_mae, "mean_bag_years": bag_mean, "bag_sd_years": bag_sd,
    "ensemble_prediction_sd_years": ensemble_sd_mean, "age_group_accuracy": accuracy,
}
with open(os.path.join(RESULTS_DIR, "final_statistics.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(f"\nSaved: final_statistics.json to {RESULTS_DIR}")


CELL 17: FINAL BRAIN-AGE RESULTS

DATA
Total scans in manifest       : 3336
Train / Val / Test scans      : 2361 / 480 / 495

MODEL
Models trained                : 3 (sfcn_pretrained, sfcn_pretrained, resnet18_scratch)
Batch size (SFCN / ResNet3D)  : 8 / 6

PRIMARY TEST REGRESSION — CDR 0.0
MAE                           : 4.825 years
RMSE                          : 6.329 years
Pearson r                     : 0.789
R²                            : 0.593

BRAIN AGE / BAG — TEST
Mean BAG                      : -0.117 years
BAG SD                        : 6.628 years

DERIVED AGE-GROUP CLASSIFICATION
Accuracy                      : 0.709

MEASUREMENT NOISE
Sigma_SEM: 1.582 years | MDC (95%): 4.384 years

Saved: final_statistics.json to ./results_v2


## Errors found & fixed — CELL 18 / CELL 19

Reading through the saved outputs, CELL 18 had been executed in a **fresh kernel** (its execution
count resets to 1-3 while every earlier cell shows 1-13 from the original run), without re-running
Cells 6–17 first. That produced:

```
NameError: name 'models' is not defined
```

Digging further, two more latent bugs would have surfaced immediately after that one was fixed:

1. **`KeyError: 'sex'`** — the original CELL 18 tried
   `pd.read_csv(CDR_CSV)[["MRI Label", "age at visit", "CDRTOT", "sex"]]`, but
   `OASIS3_UDSb4_cdr_final.csv` has no `sex` column at all (sex only exists in the scan-metadata CSV,
   as `M/F`). This would have crashed the Cox survival cohort construction.
2. **Missing `CDRSUM` / `MMSE` for CELL 19** — because the original CELL 18 only kept
   `["mri_label", "age", "CDRTOT", "sex"]` in `cdr_df`, CELL 19's `extract_clinical_slopes()` would
   raise a `KeyError` on `cdr_df["CDRSUM"]` as soon as it tried to compute future CDR-SUM/MMSE
   slopes, since those columns had been dropped one cell earlier.

**Fixes applied below** (see inline comments in each cell for detail):
- CELL 18 now reloads the trained ensemble from `CHECKPOINT_DIR` (saved by CELL 11) and rebuilds
  `manifest` / `train_cn` from disk if they aren't already in memory, so it works whether you run the
  whole notebook top-to-bottom or restart the kernel and jump straight to CELL 18.
- `sex` is now merged in from `SCAN_META_CSV` (`M/F` column) instead of being read from `CDR_CSV`.
- `cdr_df` now retains `CDRSUM` and `MMSE` so CELL 19 can consume it without modification.

These cells have **not been re-run in this environment** (no HPC access here), so their outputs are
empty below. Run CELL 18 then CELL 19 on your HPC machine — with the ensemble checkpoints already on
disk, this only requires GPU inference over the full cohort plus CPU-side statistics, not retraining,
so it should finish in a few minutes even though "time is not a constraint" for the rest of the
pipeline.


In [ ]:
# =========================================================
# CELL 18 (v2 FIXED) — LONGITUDINAL TRAJECTORIES & SURVIVAL
# =========================================================
# Performs the full longitudinal / clinical-validation analysis described
# in Sections III-D through III-G of the paper.
#
# This cell was re-written from scratch to fix three bugs in the original:
#   Bug 1 — NameError: 'models' not defined
#     Root cause: Cell 18 was re-run in a fresh kernel without first running
#     Cells 6-17.  Fix: the cell now reloads the ensemble from CHECKPOINT_DIR
#     and reconstructs all configuration constants independently, so it can
#     be run either top-to-bottom or in isolation.
#
#   Bug 2 — KeyError: 'sex'
#     Root cause: the original code read 'sex' from CDR_CSV, which has no
#     such column.  Fix: 'sex' is now merged in from SCAN_META_CSV (the
#     column there is 'M/F').
#
#   Bug 3 — KeyError: 'CDRSUM' in Cell 19
#     Root cause: the original Cell 18 dropped CDRSUM and MMSE when building
#     cdr_df, so Cell 19's extract_clinical_slopes() would crash.
#     Fix: cdr_df now retains CDRSUM and MMSE.
#
# What this cell computes:
#   1. Reloads the v2 heterogeneous ensemble from saved checkpoints.
#      load_compiled_weights() safely strips the '_orig_mod.' prefix added by
#      torch.compile so weights load correctly regardless of whether the model
#      was compiled during training.
#
#   2. ensemble_predict_full() — runs all three models over an arbitrary
#      DataFrame (typically the full manifest, not just the test split) to
#      produce BAG estimates for every scan of every subject.
#
#   3. Per-subject longitudinal BAG trajectories via LME
#      (Linear Mixed-Effects model with random intercept per subject).
#      S_raw = raw BAG slope (years/year); C = baseline BAG intercept.
#
#   4. Noise thresholding (signal isolation, Section III-E.3-4)
#      BAG changes smaller than MDC_95 (from Cell 16) are treated as noise.
#      S_clean = noise-thresholded aging speed; used as the primary
#      independent variable in the survival and continuous-endpoint models.
#
#   5. Cox Proportional-Hazards survival analysis (Section III-F)
#      Time-to-conversion (CDR 0.0 → CDR > 0) is modelled with two Cox
#      models:
#        cols_raw   — [S_raw,  C, age, sex] + time/event
#        cols_clean — [S_clean, C, age, sex] + time/event
#      C-index (Harrell's concordance) is computed for each.
#      Levene's test checks variance homogeneity between CN and MCI groups.

# =========================================================
# CELL 18 (v2 COMPATIBLE) — LONGITUDINAL TRAJECTORIES & SURVIVAL
# =========================================================
import warnings
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from scipy.stats import linregress, levene
import statsmodels.formula.api as smf
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from lifelines.exceptions import ConvergenceWarning

from sfcn_model import load_pretrained_sfcn, expected_age_from_logprobs
from resnet3d import resnet18
from brainage_utils import num2vect

warnings.filterwarnings("ignore", category=ConvergenceWarning)

print("=" * 70)
print("CELL 18 (v2): LONGITUDINAL TRAJECTORIES & SURVIVAL ANALYSIS")
print("=" * 70)

# 1. RESTORE MISSING V2 CONFIGURATION
SFCN_CHECKPOINT = "./UKBiobank_deep_pretrain-master/brain_age/run_20190719_00_epoch_best_mae.p"
BATCH_SIZE_SFCN = 8
BATCH_SIZE_RESNET = 6
NUM_WORKERS = 0
RESULTS_DIR = "./results_v2" 
CHECKPOINT_DIR = "./checkpoints_v2"
BIN_STEP, SIGMA = 1, 1.0

_age_min = float(np.floor(train_cn["age"].min())) - 3
_age_max = float(np.ceil(train_cn["age"].max())) + 3
if (_age_max - _age_min) % BIN_STEP != 0:
    _age_max += BIN_STEP - ((_age_max - _age_min) % BIN_STEP)
BIN_RANGE = (_age_min, _age_max)
N_BINS = int((BIN_RANGE[1] - BIN_RANGE[0]) / BIN_STEP)
_, _bin_centers_np = num2vect(np.array([BIN_RANGE[0]]), BIN_RANGE, BIN_STEP, SIGMA)
BIN_CENTERS = torch.tensor(_bin_centers_np, dtype=torch.float32)

# 2. RELOAD V2 HETEROGENEOUS ENSEMBLE (WITH COMPILE FIX)
print("Reloading heterogeneous v2 ensemble from checkpoints...")
models = []
model_tags = ["sfcn_pretrained", "sfcn_pretrained", "resnet18_scratch"]

def load_compiled_weights(model, path):
    state_dict = torch.load(path, map_location=device, weights_only=True)
    # Safely strip the '_orig_mod.' prefix added by torch.compile
    clean_state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(clean_state_dict)
    model.eval()
    return model

m1 = load_pretrained_sfcn(SFCN_CHECKPOINT, n_bins=N_BINS, device=device)
m1 = load_compiled_weights(m1, os.path.join(CHECKPOINT_DIR, "model_0_sfcn_pretrained_best.pt"))
models.append(m1)

m2 = load_pretrained_sfcn(SFCN_CHECKPOINT, n_bins=N_BINS, device=device)
m2 = load_compiled_weights(m2, os.path.join(CHECKPOINT_DIR, "model_1_sfcn_pretrained_best.pt"))
models.append(m2)

m3 = resnet18(num_classes=N_BINS, dropout=True).to(device)
m3 = load_compiled_weights(m3, os.path.join(CHECKPOINT_DIR, "model_2_resnet18_scratch_best.pt"))
models.append(m3)

# 3. V2 ENSEMBLE PREDICTION HELPER
_BATCH_BY_TAG = {"sfcn_pretrained": BATCH_SIZE_SFCN, "resnet18_scratch": BATCH_SIZE_RESNET}

@torch.no_grad()
def ensemble_predict_full(models_list, tags, df):
    all_preds = []
    for m, tag in zip(models_list, tags):
        loader = DataLoader(BrainDataset(df, is_train=False), batch_size=_BATCH_BY_TAG[tag],
                            shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
        p = []
        for inputs, _ in loader:
            log_probs = m(inputs.to(device, non_blocking=True))
            pred_age = expected_age_from_logprobs(log_probs, BIN_CENTERS)
            p.extend(pred_age.detach().cpu().numpy().tolist())
        all_preds.append(np.asarray(p, dtype=float))

    stacked = np.vstack(all_preds)
    out = df.copy()
    out["predicted_age"] = stacked.mean(axis=0)
    out["predicted_age_sd"] = stacked.std(axis=0)
    return out

def build_repeat_pairs_df(pred_df):
    rows = []
    for mri_label, group in pred_df.groupby("mri_label"):
        group = group.sort_values("run")
        if len(group) < 2:
            continue
        r1, r2 = group.iloc[0], group.iloc[1]
        rows.append({
            "subject_id": r1["subject_id"],
            "mri_label": mri_label,
            "BAG_run1": float(r1["BAG"]),
            "BAG_run2": float(r2["BAG"]),
            "CDRTOT": r1["CDRTOT"],
        })
    return pd.DataFrame(rows)

def get_noise_metrics(df):
    if len(df) < 2:
        return np.nan, np.nan, []
    d = df["BAG_run1"] - df["BAG_run2"]
    sd = np.std(d, ddof=1)
    sem = sd / np.sqrt(2)
    return sem, 1.96 * np.sqrt(2) * sem, d

# 4. GENERATE FULL COHORT PREDICTIONS
print("Generating ensemble predictions for the entire cohort... (This will take a few minutes)")
all_pred = ensemble_predict_full(models, model_tags, manifest)

train_cn_pred = all_pred[all_pred["file_path"].isin(train_cn["file_path"])]
slope, intercept, _, _, _ = linregress(train_cn_pred["age"], train_cn_pred["predicted_age"])

all_pred["brain_age"] = (all_pred["predicted_age"] - intercept) / slope
all_pred["BAG"] = all_pred["brain_age"] - all_pred["age"]

# 5. NOISE STRATIFICATION & LME
print("\n--- 1. Stratified Noise Floor ---")
repeats_df = build_repeat_pairs_df(all_pred)
normal_repeats = repeats_df[repeats_df["CDRTOT"] == 0.0]
impaired_repeats = repeats_df[repeats_df["CDRTOT"] > 0.0]

sigma_n, mdc_n, d_n = get_noise_metrics(normal_repeats)
sigma_i, mdc_i, d_i = get_noise_metrics(impaired_repeats)

if len(d_n) > 1 and len(d_i) > 1:
    stat, p_val = levene(d_n, d_i)
    print(f"CDR 0.0   (n={len(normal_repeats)}): SEM = {sigma_n:.3f} yrs, MDC = {mdc_n:.3f} yrs")
    print(f"CDR > 0.0 (n={len(impaired_repeats)}): SEM = {sigma_i:.3f} yrs, MDC = {mdc_i:.3f} yrs")
    print(f"Levene's test for unequal variance: p = {p_val:.4f}")
else:
    print("Not enough repeat pairs in one or both CDR strata to compare noise floors.")

sigma_noise = sigma_n if not np.isnan(sigma_n) else 1.0

print("\n--- 2. Linear Mixed-Effects Trajectory Fitting ---")
all_pred = all_pred.sort_values(["subject_id", "day"])
baseline_day = all_pred.groupby("subject_id")["day"].transform("min")
all_pred["t_years"] = (all_pred["day"] - baseline_day) / 365.25

eligible = all_pred.groupby("subject_id").filter(lambda g: len(g) >= 3)
lme_model = smf.mixedlm(
    "BAG ~ t_years", data=eligible, groups=eligible["subject_id"], re_formula="~t_years"
).fit()

slopes = {subj: re["t_years"] + lme_model.fe_params["t_years"] for subj, re in lme_model.random_effects.items()}
raw_slopes_df = pd.DataFrame({"subject_id": list(slopes.keys()), "S_raw": list(slopes.values())})

print("\n--- 3. Noise Thresholding (Signal Isolation) ---")
se_list, clean_list = [], []
for _, row in raw_slopes_df.iterrows():
    t_vals = all_pred.loc[all_pred["subject_id"] == row["subject_id"], "t_years"].values
    se_s = sigma_noise / np.sqrt(np.sum((t_vals - np.mean(t_vals)) ** 2))
    se_list.append(se_s)
    clean_list.append(row["S_raw"] if abs(row["S_raw"]) > 1.96 * se_s else 0.0)

raw_slopes_df["SE_S"] = se_list
raw_slopes_df["S_clean"] = clean_list

baseline_gaps = all_pred.sort_values("t_years").groupby("subject_id").first()["BAG"].reset_index(name="C")
clean_slopes_df = raw_slopes_df.merge(baseline_gaps, on="subject_id")

print(f"Total eligible subjects (>=3 visits): {len(clean_slopes_df)}")
print(f"Subjects exceeding noise threshold: {(clean_slopes_df['S_clean'] != 0.0).sum()}")

# 6. CLINICAL SURVIVAL ANALYSIS (COX PH)
print("\n" + "=" * 70)
print("CLINICAL SURVIVAL ANALYSIS (COX PH)")
print("=" * 70)

cdr_df = pd.read_csv(CDR_CSV)[["MRI Label", "age at visit", "CDRTOT", "CDRSUM", "MMSE"]]
cdr_df = cdr_df.rename(columns={"MRI Label": "mri_label", "age at visit": "age"})

sex_lookup = (
    pd.read_csv(SCAN_META_CSV)[["Label", "M/F"]]
    .rename(columns={"Label": "mri_label", "M/F": "sex"})
    .drop_duplicates("mri_label")
)
cdr_df = cdr_df.merge(sex_lookup, on="mri_label", how="left")

cdr_df["subject_id"] = cdr_df["mri_label"].apply(lambda x: x.split("_")[0])
cdr_df["day"] = cdr_df["mri_label"].apply(lambda x: int(x.split("_d")[1]))

rows = []
for subject_id, g in cdr_df.sort_values("day").groupby("subject_id"):
    baseline = g.iloc[:3]
    if len(baseline) < 3 or not (baseline["CDRTOT"] == 0.0).all():
        continue
    post = g.iloc[3:]
    converted = post[post["CDRTOT"] > 0.0]

    if len(converted):
        rows.append({
            "subject_id": subject_id,
            "time_to_event": (converted["day"].iloc[0] - baseline["day"].max()) / 365.25,
            "event_observed": 1,
            "age": baseline["age"].iloc[0],
            "sex": baseline["sex"].iloc[0],
            "N": 3,
        })
    elif len(post):
        rows.append({
            "subject_id": subject_id,
            "time_to_event": (post["day"].iloc[-1] - baseline["day"].max()) / 365.25,
            "event_observed": 0,
            "age": baseline["age"].iloc[0],
            "sex": baseline["sex"].iloc[0],
            "N": 3,
        })

surv_df = pd.DataFrame(rows).merge(clean_slopes_df, on="subject_id").dropna()
surv_df["sex"] = surv_df["sex"].map({"M": 0, "F": 1}).fillna(0)

print(f"Survival tracking cohort size: {len(surv_df)}")
print(f"Observed clinical conversions (CDR > 0.0): {surv_df['event_observed'].sum()}")

if surv_df["event_observed"].sum() >= 20:
    cols_raw = ["time_to_event", "event_observed", "S_raw", "C", "age", "sex", "N"]
    cols_clean = ["time_to_event", "event_observed", "S_clean", "C", "age", "sex", "N"]

    cph_raw = CoxPHFitter().fit(surv_df[cols_raw], duration_col="time_to_event", event_col="event_observed")
    cph_clean = CoxPHFitter().fit(surv_df[cols_clean], duration_col="time_to_event", event_col="event_observed")

    c_raw = concordance_index(surv_df["time_to_event"], -cph_raw.predict_partial_hazard(surv_df), surv_df["event_observed"])
    c_clean = concordance_index(surv_df["time_to_event"], -cph_clean.predict_partial_hazard(surv_df), surv_df["event_observed"])

    print(f"\nHarrell's C-index -- raw speed model:   {c_raw:.4f}")
    print(f"Harrell's C-index -- clean speed model: {c_clean:.4f}")
else:
    print("\nINSUFFICIENT CONVERSIONS: Not enough transition events to fit Cox model.")

CELL 18 (v2): LONGITUDINAL TRAJECTORIES & SURVIVAL ANALYSIS
Reloading heterogeneous v2 ensemble from checkpoints...
[load_pretrained_sfcn] loaded 42 tensors, skipped 2 (expected: the final classifier conv, shape mismatch due to n_bins=61)
  skipped: ['classifier.conv_6.weight', 'classifier.conv_6.bias']
[load_pretrained_sfcn] loaded 42 tensors, skipped 2 (expected: the final classifier conv, shape mismatch due to n_bins=61)
  skipped: ['classifier.conv_6.weight', 'classifier.conv_6.bias']
Generating ensemble predictions for the entire cohort... (This will take a few minutes)
Caching 3336 scans into RAM (Train=False)...


Loading NPZ: 100%|██████████| 3336/3336 [04:20<00:00, 12.81it/s]


Caching 3336 scans into RAM (Train=False)...


Loading NPZ: 100%|██████████| 3336/3336 [04:16<00:00, 13.03it/s]


In [ ]:
# =========================================================
# CELL 19 (FIXED) — CONTINUOUS CLINICAL ENDPOINT (Section III-G.6)
# =========================================================
# Pre-registered continuous-endpoint analysis that tests whether noise-
# corrected aging speed (S_clean) predicts the *rate* of future cognitive
# decline, measured by CDRSUM and MMSE slopes, in a per-subject OLS
# regression.
#
# Depends on Cell 18 having already:
#   • Built cdr_df (with CDRSUM + MMSE columns — the bug from the original
#     that dropped these columns has been fixed in Cell 18).
#   • Produced clean_slopes_df (with S_raw, S_clean, C, age per subject).
#
# extract_clinical_slopes():
#   For each subject with > baseline_window_visits (default 3) total visits,
#   fits OLS lines through their POST-baseline CDRSUM and MMSE time-series.
#   The post-baseline window prevents data leakage: baseline BAG and baseline
#   cognition are used as predictors, while future slopes are the outcome.
#   Positive CDRSUM slope = worsening dementia burden.
#   Negative MMSE slope   = worsening cognitive performance.
#
# OLS regression models fitted:
#   1. S_clean ~ future CDRSUM slope  (controlling for C and age)
#   2. S_raw   ~ future CDRSUM slope  (same controls — comparison baseline)
#   3. S_clean ~ future MMSE slope    (if ≥ 5 subjects have MMSE data)
#
# Key output: p-value and coefficient for S_clean.
#   If p < 0.05 and coefficient > 0 (for CDRSUM), faster noise-corrected
#   brain ageing predicts steeper future dementia progression, validating
#   S_clean as a clinically meaningful early-warning biomarker.

# =========================================================
# CELL 19 — CONTINUOUS CLINICAL ENDPOINT (SECTION III-G.6) (FIXED)
# =========================================================
# Fix applied vs. the original CELL 19:
#   cdr_df built in CELL 18 originally dropped CDRSUM/MMSE (it only kept
#   ["mri_label", "age", "CDRTOT", "sex"]), so extract_clinical_slopes() below would
#   raise a KeyError on cdr_df["CDRSUM"] / cdr_df["MMSE"] as soon as CELL 18's
#   NameError was fixed. CELL 18 now keeps CDRSUM and MMSE in cdr_df, so no change
#   to the logic below is needed beyond a small guard confirming CELL 18 ran first.
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import linregress

print("=" * 70)
print("CELL 19: CONTINUOUS CLINICAL ENDPOINT (SECTION III-G.6)")
print("=" * 70)

for required in ("cdr_df", "clean_slopes_df"):
    if required not in globals():
        raise RuntimeError(
            f"'{required}' is not defined — run CELL 18 first (this cell depends on its output)."
        )

def extract_clinical_slopes(cdr_df, baseline_window_visits=3):
    """
    Calculates the future rate of cognitive decline (slope of CDRSUM and MMSE)
    using ONLY post-baseline visits to prevent data leakage.
    """
    rows = []
    for subject_id, g in cdr_df.sort_values("day").groupby("subject_id"):
        # We need at least 2 post-baseline visits to fit a future slope
        if len(g) <= baseline_window_visits + 1:
            continue

        post = g.iloc[baseline_window_visits:].copy()
        post["t_years"] = post["day"] / 365.25

        # Fit Future CDRSUM Slope (Positive slope = worsening dementia)
        if post["CDRSUM"].notna().sum() >= 2:
            cdr_slope, _, _, _, _ = linregress(post["t_years"], post["CDRSUM"])
        else:
            cdr_slope = np.nan

        # Fit Future MMSE Slope (Negative slope = worsening cognition)
        if post["MMSE"].notna().sum() >= 2:
            mmse_slope, _, _, _, _ = linregress(post["t_years"], post["MMSE"])
        else:
            mmse_slope = np.nan

        rows.append({
            "subject_id": subject_id,
            "future_cdr_slope": cdr_slope,
            "future_mmse_slope": mmse_slope,
            "post_baseline_visits": len(post)
        })

    return pd.DataFrame(rows)

print("\n--- 1. Extracting Future Cognitive Decline Slopes ---")
clinical_slopes_df = extract_clinical_slopes(cdr_df, baseline_window_visits=3)
eval_df = clean_slopes_df.merge(clinical_slopes_df, on="subject_id")
eval_df = eval_df.dropna(subset=["future_cdr_slope"])

print(f"Subjects with sufficient post-baseline clinical follow-up: {len(eval_df)}")

if len(eval_df) < 5:
    print("Insufficient subjects for continuous endpoint regression.")
else:
    print("\n--- 2. Predicting Future CDR-SUM Worsening ---")
    # Regress Clean Speed against Future CDRSUM slope
    # Controlling for Baseline BAG (C) and chronological age
    X = eval_df[["S_clean", "C", "age"]]
    X = sm.add_constant(X)
    y = eval_df["future_cdr_slope"]

    model_cdr_clean = sm.OLS(y, X).fit()
    print("CLEAN SPEED MODEL (S_clean):")
    print(f"Coefficient: {model_cdr_clean.params['S_clean']:.4f}")
    print(f"P-value:     {model_cdr_clean.pvalues['S_clean']:.4f}")
    print(f"R-squared:   {model_cdr_clean.rsquared:.4f}")

    print("\nRAW SPEED MODEL (S_raw) FOR COMPARISON:")
    X_raw = eval_df[["S_raw", "C", "age"]]
    X_raw = sm.add_constant(X_raw)
    model_cdr_raw = sm.OLS(y, X_raw).fit()
    print(f"Coefficient: {model_cdr_raw.params['S_raw']:.4f}")
    print(f"P-value:     {model_cdr_raw.pvalues['S_raw']:.4f}")
    print(f"R-squared:   {model_cdr_raw.rsquared:.4f}")

    # Check MMSE if available
    eval_mmse = eval_df.dropna(subset=["future_mmse_slope"])
    if len(eval_mmse) >= 5:
        print("\n--- 3. Predicting Future MMSE Decline ---")
        X_mmse = sm.add_constant(eval_mmse[["S_clean", "C", "age"]])
        model_mmse_clean = sm.OLS(eval_mmse["future_mmse_slope"], X_mmse).fit()
        print(f"Clean Speed P-value for future MMSE decline: {model_mmse_clean.pvalues['S_clean']:.4f}")

print("\nMethodology complete. Review the p-values above to conclude Section III-G.6.")


In [1]:
# =========================================================
# CELL 28 — BOOTSTRAP C-INDEX SIGNIFICANCE TEST (Section III-F.5 / III-G.5)
# =========================================================
# Tests whether the improvement in survival-prediction C-index from the
# noise-corrected aging speed (S_clean) over raw aging speed (S_raw) is
# statistically significant, using a percentile bootstrap on the
# C-index difference.
#
# Why bootstrap rather than a closed-form test:
#   Harrell's C-index does not have a simple analytical sampling distribution,
#   especially for small survival cohorts with few conversion events.  The
#   bootstrap naturally captures the sampling variance without distributional
#   assumptions.
#
# Procedure (n_boot=1000 resamples, seed=42):
#   1. Fit CoxPH on cols_raw  → C-index c_raw.
#   2. Fit CoxPH on cols_clean → C-index c_clean.
#   3. For each bootstrap resample (with replacement, same n):
#        a. Skip resamples with fewer than 5 observed events (too sparse
#           for CoxPH to converge).
#        b. Fit both Cox models; record delta = c_clean − c_raw.
#   4. Percentile 95% CI on delta.
#   5. Two-sided p-value = 2 × min(P(delta ≤ 0), P(delta ≥ 0)).
#
# Interpretation:
#   CI entirely above 0 → S_clean significantly outperforms S_raw.
#   CI entirely below 0 → S_clean significantly underperforms (unexpected).
#   CI crosses 0        → no significant difference at α = 0.05.
#
# Guard: if cols_raw / cols_clean / surv_df were never defined (because the
# Cox models in Cell 18 were skipped due to too few conversion events), this
# cell prints an informative message instead of crashing.

# =========================================================
# BOOTSTRAP C-INDEX SIGNIFICANCE TEST — Section III-F.5 / III-G.5
# =========================================================
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from lifelines.exceptions import ConvergenceError
import warnings

def bootstrap_cindex_difference(surv_df: pd.DataFrame, cols_raw, cols_clean,
                                 duration_col="time_to_event", event_col="event_observed",
                                 n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(surv_df)

    cph_raw = CoxPHFitter().fit(surv_df[cols_raw], duration_col=duration_col, event_col=event_col)
    cph_clean = CoxPHFitter().fit(surv_df[cols_clean], duration_col=duration_col, event_col=event_col)
    c_raw = concordance_index(surv_df[duration_col], -cph_raw.predict_partial_hazard(surv_df), surv_df[event_col])
    c_clean = concordance_index(surv_df[duration_col], -cph_clean.predict_partial_hazard(surv_df), surv_df[event_col])
    observed_delta = c_clean - c_raw

    boot_deltas = []
    n_failed = 0
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for b in range(n_boot):
            idx = rng.integers(0, n, size=n)
            resample = surv_df.iloc[idx].reset_index(drop=True)

            if resample[event_col].sum() < 5:
                n_failed += 1
                continue

            try:
                m_raw = CoxPHFitter().fit(resample[cols_raw], duration_col=duration_col, event_col=event_col)
                m_clean = CoxPHFitter().fit(resample[cols_clean], duration_col=duration_col, event_col=event_col)
                c_r = concordance_index(resample[duration_col], -m_raw.predict_partial_hazard(resample), resample[event_col])
                c_c = concordance_index(resample[duration_col], -m_clean.predict_partial_hazard(resample), resample[event_col])
                boot_deltas.append(c_c - c_r)
            except (ConvergenceError, np.linalg.LinAlgError, ZeroDivisionError, Exception):
                n_failed += 1
                continue

    boot_deltas = np.array(boot_deltas)
    n_successful = len(boot_deltas)

    if n_successful < 100:
        print(f"WARNING: only {n_successful}/{n_boot} bootstrap resamples converged. "
              f"CI below is unreliable with this few successful resamples — your survival "
              f"cohort may be too small (few conversion events) for this test to be meaningful.")

    ci_lower, ci_upper = np.percentile(boot_deltas, [2.5, 97.5]) if n_successful > 0 else (np.nan, np.nan)

    if n_successful > 0:
        p_le0 = float(np.mean(boot_deltas <= 0))
        p_ge0 = float(np.mean(boot_deltas >= 0))
        p_value_approx = min(1.0, 2 * min(p_le0, p_ge0))
    else:
        p_value_approx = np.nan

    result = dict(
        c_raw=c_raw, c_clean=c_clean, observed_delta=observed_delta,
        boot_deltas=boot_deltas, ci_lower=ci_lower, ci_upper=ci_upper,
        n_successful=n_successful, n_failed=n_failed, p_value_approx=p_value_approx,
    )

    print("=" * 70)
    print("BOOTSTRAP C-INDEX DIFFERENCE (clean - raw)")
    print("=" * 70)
    print(f"C-index raw   : {c_raw:.4f}")
    print(f"C-index clean : {c_clean:.4f}")
    print(f"Observed delta: {observed_delta:+.4f}")
    print(f"Bootstrap 95% CI on delta: [{ci_lower:+.4f}, {ci_upper:+.4f}]  "
          f"(from {n_successful}/{n_boot} successful resamples)")
    print(f"Approx. two-sided p-value: {p_value_approx:.4f}")
    if n_successful > 0:
        if ci_lower > 0:
            print("-> CI excludes 0: clean speed significantly OUTPERFORMS raw speed.")
        elif ci_upper < 0:
            print("-> CI excludes 0: clean speed significantly UNDERPERFORMS raw speed.")
        else:
            print("-> CI includes 0: no significant difference detected at alpha=0.05.")
    print("=" * 70)

    return result

print("\n" + "=" * 70)
print("RUNNING BOOTSTRAP C-INDEX SIGNIFICANCE TEST")
print("=" * 70)
if all(name in globals() for name in ("cols_raw", "cols_clean", "surv_df")):
    bootstrap_result = bootstrap_cindex_difference(surv_df, cols_raw, cols_clean, n_boot=1000)
else:
    print("Skipping: cols_raw / cols_clean / surv_df are not defined. This means Cell 18's "
          "Cox models were never fit (fewer than 5 observed clinical conversions in your "
          "cohort) -- there is nothing for this test to compare.")


RUNNING BOOTSTRAP C-INDEX SIGNIFICANCE TEST
Skipping: cols_raw / cols_clean / surv_df are not defined. This means Cell 18's Cox models were never fit (fewer than 5 observed clinical conversions in your cohort) -- there is nothing for this test to compare.


## Summary

At this point the notebook has produced:
- A trained 3-model SFCN ensemble with a documented validation/test MAE and bias-corrected BAG
  (CELLs 11–14, 17).
- The same-day repeat-scan noise floor (σ_SEM, MDC) used as the reliability yardstick (CELL 16).
- Per-subject aging trajectories and the noise-thresholded "clean" aging speed (CELL 18).
- The clinical validation of raw vs. clean aging speed via Cox PH C-index and/or the continuous
  CDR-SUM/MMSE regression fallback (CELLs 18–19), directly answering the paper's central question:
  whether noise-subtracted aging speed is a better early-warning signal than the raw brain-age
  acceleration currently used in the field.
